# Invoice OCR — free / offline

Extract text from an invoice image using **RapidOCR** (ONNX + PaddleOCR models via pip — no API key, no system binary, runs on CPU offline).

Every line keeps its **confidence + bounding box**, so any extracted value is traceable to where it sat on the page.

In [10]:
# 1. Install (run once)
%pip install -q rapidocr_onnxruntime opencv-python numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# 2. OCR logic
from __future__ import annotations
from dataclasses import dataclass, field
from pathlib import Path
import cv2
import numpy as np
from rapidocr_onnxruntime import RapidOCR

_engine = RapidOCR()  # loads ONNX models once


@dataclass
class OcrLine:
    text: str
    confidence: float
    bbox: list  # 4 corner points [[x, y], ...]

    @property
    def top(self):
        return min(p[1] for p in self.bbox)

    @property
    def left(self):
        return min(p[0] for p in self.bbox)


@dataclass
class OcrResult:
    full_text: str
    lines: list = field(default_factory=list)
    mean_confidence: float = 0.0


def _preprocess(path: Path) -> np.ndarray:
    """Read image (unicode-safe) and upscale small images so small fonts stay legible."""
    img = cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f'Could not read image: {path}')
    h, w = img.shape[:2]
    if max(h, w) < 1600:
        scale = 1600 / max(h, w)
        img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    return img


def extract_text(image_path, preprocess: bool = True) -> OcrResult:
    """Run OCR on an invoice image and return ordered text + per-line provenance."""
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(path)

    img = _preprocess(path) if preprocess else str(path)
    raw, _elapsed = _engine(img)
    if not raw:
        return OcrResult(full_text='', lines=[], mean_confidence=0.0)

    lines = [
        OcrLine(text=str(text).strip(), confidence=float(score), bbox=box)
        for box, text, score in raw
        if str(text).strip()
    ]
    # Reading order: group into rows by vertical position, then left-to-right.
    lines.sort(key=lambda ln: (round(ln.top / 15), ln.left))

    full_text = '\n'.join(ln.text for ln in lines)
    mean_conf = sum(ln.confidence for ln in lines) / len(lines) if lines else 0.0
    return OcrResult(full_text=full_text, lines=lines, mean_confidence=mean_conf)

In [12]:
# 3. Run it on an invoice image
img = next(iter(sorted(Path('data').glob('*.png'))))  # or set your own path
result = extract_text(img)
print(f'{img.name}: {len(result.lines)} lines | mean confidence {result.mean_confidence:.1%}')

ChatGPT Image Jun 19, 2026, 08_16_23 PM.png: 54 lines | mean confidence 79.2%


In [13]:
# 4. Full recognized text, in reading order
print(result.full_text)

GLOBAL
TAXINVOICE
SOLUTIONS
ImvoiceNoINV-2024-0756
321BusineesPark
IvoiceDate15-Apr-2024
Bangalote-560078,inda
GSTIN:29ABCDE1234F1Z5
Due Date
15-May-2024
BILL TO
SHIP TO
TechinnovationsPMLid
Tech Innovations PvtLtd
45,lndustnalArea,Phase2
Warehouse12.ndustrialArea
Pue-411057,Maharashtna
Phase2.Pue-411057
GSTIN:27FGHUK6789L1Z2
Maharashtra
DESCRIPTION
HSNSAC
OTY
UNITPRICE
AMOUNT
HPLaptp158CBRAM512GBSSD
84713010
45,000.00
4,50,000.00
Wireless Mouse Logitech M221
84716060
10
1,100.00
11.000.00
LaptopBagTargus15.6inch
42021250
850.00
8,500.00
SUBTOTAL
4,69,500.00
CGST09%
42.255.00
SOSTe%
42.255.00
TOTAL
5,54,010.00
Armount in Words
RupeesFveLahFyFeoThousand Ten Orty
For Global Solutions
Terms&Condtions
Poytwe dn fon olcedate
2ueret0tospa
bedhapdoverduepyment
Authorized Signatory


In [14]:
# 5. Per-line provenance: confidence + bbox top-left corner + text
for ln in result.lines:
    print(f'[{ln.confidence:5.1%}] ({ln.left:5.0f},{ln.top:5.0f})  {ln.text}')

[75.6%] (  240,   74)  GLOBAL
[87.4%] ( 1070,   72)  TAXINVOICE
[88.1%] (  242,  126)  SOLUTIONS
[88.8%] ( 1060,  166)  ImvoiceNoINV-2024-0756
[81.8%] (  125,  189)  321BusineesPark
[87.3%] ( 1059,  205)  IvoiceDate15-Apr-2024
[74.8%] (  124,  227)  Bangalote-560078,inda
[91.4%] (  121,  262)  GSTIN:29ABCDE1234F1Z5
[82.9%] ( 1059,  251)  Due Date
[85.5%] ( 1190,  251)  15-May-2024
[77.1%] (  124,  331)  BILL TO
[80.7%] (  869,  341)  SHIP TO
[84.1%] (  127,  373)  TechinnovationsPMLid
[84.6%] (  872,  386)  Tech Innovations PvtLtd
[74.8%] (  125,  407)  45,lndustnalArea,Phase2
[85.2%] (  871,  423)  Warehouse12.ndustrialArea
[76.6%] (  123,  443)  Pue-411057,Maharashtna
[87.5%] (  870,  457)  Phase2.Pue-411057
[82.9%] (  121,  482)  GSTIN:27FGHUK6789L1Z2
[81.0%] (  869,  491)  Maharashtra
[89.4%] (  365,  566)  DESCRIPTION
[83.5%] (  773,  566)  HSNSAC
[65.6%] (  951,  564)  OTY
[79.7%] ( 1084,  564)  UNITPRICE
[85.4%] ( 1355,  565)  AMOUNT
[78.2%] (  188,  610)  HPLaptp158CBRAM512GBSS

## PDF extraction

PDFs have two flavours:
- **Native** (has a text layer) → read text directly with PyMuPDF (`fitz`). Exact, instant, no OCR.
- **Scanned** (image-only) → render each page to an image and run the same OCR path above.

`extract_document()` auto-detects which, page by page.

In [15]:
# 6. Unified PDF + image extraction
import fitz  # PyMuPDF

_MIN_CHARS_PER_PAGE = 50  # below this a page is treated as scanned → OCR


def _ocr_pdf_page(page, dpi=200) -> OcrResult:
    """Render a scanned PDF page to an image and OCR it."""
    pix = page.get_pixmap(matrix=fitz.Matrix(dpi / 72, dpi / 72))
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR if pix.n == 3 else cv2.COLOR_RGBA2BGR)
    raw, _ = _engine(img)
    lines = [OcrLine(str(t).strip(), float(s), b) for b, t, s in (raw or []) if str(t).strip()]
    lines.sort(key=lambda ln: (round(ln.top / 15), ln.left))
    text = '\n'.join(ln.text for ln in lines)
    conf = sum(ln.confidence for ln in lines) / len(lines) if lines else 0.0
    return OcrResult(full_text=text, lines=lines, mean_confidence=conf)


def extract_document(path):
    """Extract text from a PDF (native text layer or scanned) or an image file."""
    path = Path(path)
    if path.suffix.lower() in {'.png', '.jpg', '.jpeg', '.tiff', '.tif', '.webp'}:
        res = extract_text(path)
        return {'source': 'image-ocr', 'mean_confidence': res.mean_confidence,
                'pages': [{'page': 1, 'mode': 'ocr', 'text': res.full_text,
                           'confidence': res.mean_confidence}]}

    doc = fitz.open(str(path))
    pages, any_ocr = [], False
    for i, page in enumerate(doc):
        text = page.get_text().strip()
        if len(text) >= _MIN_CHARS_PER_PAGE:
            pages.append({'page': i + 1, 'mode': 'native', 'text': text, 'confidence': 1.0})
        else:
            any_ocr = True
            res = _ocr_pdf_page(page)
            pages.append({'page': i + 1, 'mode': 'ocr', 'text': res.full_text,
                          'confidence': res.mean_confidence})
    doc.close()
    confs = [p['confidence'] for p in pages]
    return {'source': 'pdf-mixed' if any_ocr else 'pdf-native',
            'mean_confidence': sum(confs) / len(confs) if confs else 0.0,
            'pages': pages}

In [16]:
# 7. Extract every PDF in data/
for pdf in sorted(Path('data').glob('*.pdf')):
    doc = extract_document(pdf)
    print(f"{pdf.name:35s} {doc['source']:11s} conf={doc['mean_confidence']:.0%}  pages={len(doc['pages'])}")

invoice_01_happy.pdf                pdf-native  conf=100%  pages=1
invoice_02_amount_mismatch.pdf      pdf-native  conf=100%  pages=1
invoice_03_missing_po.pdf           pdf-native  conf=100%  pages=1
invoice_04_duplicate.pdf            pdf-native  conf=100%  pages=1
invoice_05_split_po.pdf             pdf-native  conf=100%  pages=1
invoice_06_tax_error.pdf            pdf-native  conf=100%  pages=1


In [17]:
# 8. Show extracted text from one PDF
doc = extract_document('data/invoice_01_happy.pdf')
print(f"source={doc['source']} | mean confidence={doc['mean_confidence']:.0%}\n")
for pg in doc['pages']:
    print(f"--- page {pg['page']} ({pg['mode']}, conf={pg['confidence']:.0%}) ---")
    print(pg['text'])

source=pdf-native | mean confidence=100%

--- page 1 (native, conf=100%) ---
VENDOR INVOICE
Vendor: ABC Technologies
Invoice Number: INV-1001
PO Number: PO-456
Subtotal: INR 50000
Tax: INR 9000
Total: INR 59000


## Alternative engine: PaddleOCR

RapidOCR runs the *same Paddle models* via ONNX (light, ~zero setup). Full **PaddleOCR** is heavier (downloads `paddlepaddle` + models on first run) but tends to recover word spacing and stylized fonts better.

```bash
pip install --user paddlepaddle paddleocr
```

> On Windows with `paddlepaddle` 3.x, inference can crash in the oneDNN backend (`ConvertPirAttribute2RuntimeAttribute`). The fix is `enable_mkldnn=False` + the `FLAGS_use_mkldnn=0` env var set **before** importing paddle (done in the cell below).

In [18]:
# 9. PaddleOCR engine (mkldnn disabled to avoid the Windows oneDNN crash)
import os
os.environ['FLAGS_use_mkldnn'] = '0'  # must be set before paddle is imported

from paddleocr import PaddleOCR

_paddle = PaddleOCR(
    lang='en',
    enable_mkldnn=False,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)


def extract_text_paddle(image_path, preprocess: bool = True) -> OcrResult:
    """Same contract as extract_text(), but backed by full PaddleOCR."""
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(path)

    img = _preprocess(path) if preprocess else cv2.imdecode(
        np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_COLOR)

    res = _paddle.predict(img)
    if not res:
        return OcrResult(full_text='', lines=[], mean_confidence=0.0)

    r = res[0]
    polys = r.get('rec_polys', r.get('dt_polys', []))
    lines = [
        OcrLine(text=str(t).strip(), confidence=float(s),
                bbox=[[float(x), float(y)] for x, y in poly])
        for poly, t, s in zip(polys, r['rec_texts'], r['rec_scores'])
        if str(t).strip()
    ]
    lines.sort(key=lambda ln: (round(ln.top / 15), ln.left))
    full_text = '\n'.join(ln.text for ln in lines)
    mean_conf = sum(ln.confidence for ln in lines) / len(lines) if lines else 0.0
    return OcrResult(full_text=full_text, lines=lines, mean_confidence=mean_conf)

c:\Users\Rohit\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Rohit\AppData\Local\Programs\Python\Python313\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Rohit\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually:

In [19]:
# 10. PaddleOCR result + side-by-side comparison with RapidOCR
paddle_res = extract_text_paddle(img)
print(f'PaddleOCR: {len(paddle_res.lines)} lines | mean confidence {paddle_res.mean_confidence:.1%}\n')
print(paddle_res.full_text)

PaddleOCR: 61 lines | mean confidence 98.6%

GLOBAL
TAX INVOICE
SOLUTIONS
Invoice No
INV-2024-0756
321 Business Park
Invoice Date
15-Apr-2024
Bangalore-560078, India
15-May-2024
GSTIN: 29ABCDE1234F1Z5
Due Date
BILL TO
SHIP TO
Tech Innovations Pvt Ltd.
Tech Innovations Pvt Ltd.
45, Industrial Area, Phase 2
Warehouse 12, Industrial Area
Pune-411057, Maharashtra
Phase 2. Pune -411057
GSTIN: 27FGHJK6789L1Z2
Maharashtra
DESCRIPTION
HSN/SAC
QTY
UNIT PRICE
AMOUNT
1
HP Laptop 15s, 8GB RAM, 512GB SSD
84713010
10
45,000.00
4,50,000.00
2
Wineless Mouse Logitech M221
84716060
10
1,100.00
11,000.00
3
Laptop Bag Targus 15.6 inch
42021250
10
850.00
8,500.00
SUBTOTAL
4,69,500.00
CGST @ 9%
42,255.00
SGST 0 9%
42,255.00
TOTAL
5,54,010.00
Amount in Words:
Rupees Five Lakh Fifty Four Thousand Ten Only
For Global Solutions
Terms & Conditions
Neoper
1. Payment within 30 days from invoice date
2. interest @ 18% p.a wilt be chargedf on overdue payments.
Authorized Signatory


In [20]:
# 11. Quick quality comparison on the same image
rapid = extract_text(img)
print(f"{'metric':22s} {'RapidOCR':>12s} {'PaddleOCR':>12s}")
print(f"{'lines detected':22s} {len(rapid.lines):>12d} {len(paddle_res.lines):>12d}")
print(f"{'mean confidence':22s} {rapid.mean_confidence:>11.1%} {paddle_res.mean_confidence:>11.1%}")

metric                     RapidOCR    PaddleOCR
lines detected                   54           61
mean confidence              79.2%       98.6%


## Structured extraction with Claude

OCR gives raw text; this step turns it into **typed invoice fields** (invoice #, PO, amounts, dates, line items) — each with a confidence score and the evidence snippet it came from.

We pin the model's output to a **Pydantic schema** via `client.messages.parse()`, so the result is validated and typed (no fragile JSON parsing). Low-confidence fields are flagged for review — exactly the provenance model used in `agents/extraction.py`.

```bash
pip install anthropic pydantic
export ANTHROPIC_API_KEY=sk-ant-...   # Windows: setx ANTHROPIC_API_KEY ...
```

> Feed it the **PaddleOCR** text (cleaner) for best results. Garbled OCR → garbled fields.

In [21]:
# 12. Schema + extraction function (Claude structured outputs)
import os
import anthropic
from pydantic import BaseModel, Field

_MODEL = 'claude-opus-4-8'  # most accurate; use 'claude-haiku-4-5' for a cheaper/faster pass


class ExtractedField(BaseModel):
    value: str | None = Field(description='Value exactly as it appears, or null if absent')
    confidence: float = Field(description='0.0-1.0 confidence this value is correct')
    evidence: str | None = Field(description='The text snippet supporting this value')


class LineItem(BaseModel):
    description: ExtractedField
    quantity: ExtractedField
    unit_price: ExtractedField
    line_total: ExtractedField


class InvoiceFields(BaseModel):
    vendor_name: ExtractedField
    invoice_number: ExtractedField
    invoice_date: ExtractedField
    due_date: ExtractedField
    po_numbers: list[ExtractedField]
    currency: ExtractedField
    subtotal: ExtractedField
    tax_total: ExtractedField
    total_amount: ExtractedField
    line_items: list[LineItem]


_SYSTEM = (
    'You are an accounts-payable analyst. Extract invoice fields from OCR text. '
    'Copy values exactly as written; do not normalise or compute. If a field is '
    'absent set value to null and confidence to 0.0. confidence reflects both how '
    'clearly the value is labelled and how clean the OCR text looks.'
)


def extract_fields(ocr_text: str, model: str = _MODEL) -> InvoiceFields:
    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
    resp = client.messages.parse(
        model=model,
        max_tokens=2048,
        system=_SYSTEM,
        messages=[{'role': 'user', 'content': f'OCR TEXT:\n\n{ocr_text}'}],
        output_format=InvoiceFields,
    )
    return resp.parsed_output

In [ ]:
# 13. Run extraction on the OCR text and show fields + confidence
ocr_text = paddle_res.full_text  # cleaner than RapidOCR; or use result.full_text
fields = extract_fields(ocr_text)

_REVIEW = 0.80  # below this → flag for human review

def show(label, ef):
    flag = '  ⚠ REVIEW' if ef.confidence < _REVIEW else ''
    print(f'{label:16s} {str(ef.value):<28} [{ef.confidence:4.0%}]{flag}')

show('Vendor', fields.vendor_name)
show('Invoice #', fields.invoice_number)
show('Invoice date', fields.invoice_date)
show('Due date', fields.due_date)
show('Currency', fields.currency)
show('Subtotal', fields.subtotal)
show('Tax', fields.tax_total)
show('Total', fields.total_amount)
for i, po in enumerate(fields.po_numbers):
    show(f'PO #{i+1}', po)

print('\nLine items:')
for li in fields.line_items:
    print(f"  {str(li.description.value):40s} qty={li.quantity.value:>4} "
          f"@ {li.unit_price.value:>10} = {li.line_total.value:>12}")

## Layout reconstruction from bounding boxes

OCR returns each text fragment with a bounding box. By grouping fragments whose **vertical** positions overlap into *rows*, then ordering each row by **horizontal** position, we recover the page's 2-D structure — left vs. right cells on the same line (e.g. `Invoice No` … `INV-2024-0756`, or the line-item columns).

`build_layout()` returns a structured schema (rows → cells with x/y/text) and can render it as spaced ASCII that mirrors the original placement. Works best on PaddleOCR boxes.

In [22]:
# 14. Reconstruct page layout from OCR bounding boxes
from statistics import median


def _y_bounds(ln):
    ys = [p[1] for p in ln.bbox]
    return min(ys), max(ys)


def build_layout(lines, row_tol_frac: float = 0.6):
    """Group OcrLines into rows (by vertical overlap), each row ordered left→right.

    Returns: list of rows; each row is a dict with 'y' and 'cells',
    where each cell = {'text', 'x0', 'x1', 'y'}.
    """
    if not lines:
        return []

    # Typical line height → vertical tolerance for "same row"
    heights = [b - t for t, b in (_y_bounds(l) for l in lines)]
    tol = median(heights) * row_tol_frac

    rows = []
    for ln in sorted(lines, key=lambda l: sum(_y_bounds(l)) / 2):
        yc = sum(_y_bounds(ln)) / 2
        x0 = min(p[0] for p in ln.bbox)
        x1 = max(p[0] for p in ln.bbox)
        cell = {'text': ln.text, 'x0': x0, 'x1': x1, 'y': yc}
        if rows and abs(yc - rows[-1]['y']) <= tol:
            rows[-1]['cells'].append(cell)
            # running mean keeps the row anchor stable as cells are added
            rows[-1]['y'] = sum(c['y'] for c in rows[-1]['cells']) / len(rows[-1]['cells'])
        else:
            rows.append({'y': yc, 'cells': [cell]})

    for r in rows:
        r['cells'].sort(key=lambda c: c['x0'])
    return rows


def render_layout(rows, width: int = 100) -> str:
    """ASCII render: place each cell at a column proportional to its x position."""
    max_x = max((c['x1'] for r in rows for c in r['cells']), default=1)
    scale = (width - 1) / max_x
    out = []
    for r in rows:
        buf = [' '] * width
        for c in r['cells']:
            col = int(c['x0'] * scale)
            for i, ch in enumerate(c['text']):
                if col + i < width and buf[col + i] == ' ':
                    buf[col + i] = ch
        out.append(''.join(buf).rstrip())
    return '\n'.join(out)

In [ ]:
# 15. Build the layout and show both the ASCII render and the row/cell schema
rows = build_layout(paddle_res.lines)  # PaddleOCR boxes are cleanest
print('=== ASCII layout ===')
print(render_layout(rows, width=100))

print('\n=== Row/cell schema (first 12 rows) ===')
for r in rows[:12]:
    cells = ' | '.join(f"{c['text']}@x{c['x0']:.0f}" for c in r['cells'])
    print(f"y={r['y']:6.0f}  {cells}")

[{'y': 99.0, 'cells': [{'text': 'GLOBAL', 'x0': 238.0, 'x1': 470.0, 'y': 100.0}, {'text': 'TAX INVOICE', 'x0': 1062.0, 'x1': 1486.0, 'y': 98.0}]}, {'y': 147.0, 'cells': [{'text': 'SOLUTIONS', 'x0': 239.0, 'x1': 480.0, 'y': 147.0}]}, {'y': 180.0, 'cells': [{'text': 'Invoice No', 'x0': 1058.0, 'x1': 1193.0, 'y': 180.5}, {'text': 'INV-2024-0756', 'x0': 1277.0, 'x1': 1477.0, 'y': 179.5}]}, {'y': 216.33333333333334, 'cells': [{'text': '321 Business Park', 'x0': 123.0, 'x1': 354.0, 'y': 205.5}, {'text': 'Invoice Date', 'x0': 1057.0, 'x1': 1215.0, 'y': 221.0}, {'text': '15-Apr-2024', 'x0': 1278.0, 'x1': 1440.0, 'y': 222.5}]}, {'y': 241.5, 'cells': [{'text': 'Bangalore-560078, India', 'x0': 120.0, 'x1': 444.0, 'y': 241.5}]}, {'y': 269.8333333333333, 'cells': [{'text': 'GSTIN: 29ABCDE1234F1Z5', 'x0': 119.0, 'x1': 468.0, 'y': 278.0}, {'text': 'Due Date', 'x0': 1058.0, 'x1': 1180.0, 'y': 265.5}, {'text': '15-May-2024', 'x0': 1277.0, 'x1': 1452.0, 'y': 266.0}]}, {'y': 352.0, 'cells': [{'text': 'BI